In [ ]:

!pip install pandas scikit-learn matplotlib seaborn --quiet

import pandas as pd
import os
import zipfile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files


uploaded = files.upload()


zip_name = list(uploaded.keys())[0]
extract_path = '/content/weather_data'
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

csv_files = [f for f in os.listdir(extract_path) if f.endswith('.csv')]
print("CSV files found:", csv_files)


csv_path = os.path.join(extract_path, csv_files[0])
data = pd.read_csv(csv_path)
print("Data preview:")
display(data.head())


data_encoded = data.copy()
text_columns = ['Cloud Cover', 'Season', 'Location']
for col in text_columns:
    le_col = LabelEncoder()
    data_encoded[col] = le_col.fit_transform(data_encoded[col])


le_target = LabelEncoder()
data_encoded['Weather Type'] = le_target.fit_transform(data_encoded['Weather Type'])


X = data_encoded.drop('Weather Type', axis=1)
y = data_encoded['Weather Type']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le_target.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


feat_importances = pd.Series(clf.feature_importances_, index=X.columns)
feat_importances.sort_values().plot(kind='barh', figsize=(8,5), title='Feature Importance')
plt.show()


In [ ]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier


algorithms = [
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=5)),
    ('SVM', SVC(random_state=42)),
    ('Logistic Regression', LogisticRegression(max_iter=500, random_state=42)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=42))
]


results = []

for name, model in algorithms:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({
        'Algorithm': name,
        'Accuracy': acc,
        'Precision (macro)': classification_report(y_test, y_pred, output_dict=True)['macro avg']['precision'],
        'Recall (macro)': classification_report(y_test, y_pred, output_dict=True)['macro avg']['recall'],
        'F1-score (macro)': classification_report(y_test, y_pred, output_dict=True)['macro avg']['f1-score']
    })


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Accuracy', ascending=False)
results_df


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
